In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([19, 57, 29, 72, 55,  9, 34, 61, 22, 60, 60, 60, 30, 29, 72])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.array((arr.size, label))
    one_hot[arr.shape[0], arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [5]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [6]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [7]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[19 57 29 72 55  9 34 61 22 60]
 [54 81 47 61 55 57 29 55 61 29]
 [ 9 47 50 61 81 34 61 29 61 70]
 [54 61 55 57  9 61  3 57 80  9]
 [61 54 29  0 61 57  9 34 61 55]
 [ 3 10 54 54 80 81 47 61 29 47]
 [61 64 47 47 29 61 57 29 50 61]
 [41 15 17 81 47 54 32 44 28 61]]
y
 [[57 29 72 55  9 34 61 22 60 60]
 [81 47 61 55 57 29 55 61 29 55]
 [47 50 61 81 34 61 29 61 70 81]
 [61 55 57  9 61  3 57 80  9 70]
 [54 29  0 61 57  9 34 61 55  9]
 [10 54 54 80 81 47 61 29 47 50]
 [64 47 47 29 61 57 29 50 61 54]
 [15 17 81 47 54 32 44 28 61 67]]


In [9]:
class CharNN(nn.Module):    
    def __init__(self, tokens, n_layer=2, n_hidden=256, drop_prob=0.5, lr=0.001):
        super().__init__()

        self.n_layer = n_layer
        self.n_hidden = n_hidden
        self.lr = lr

        self.chars = tokens
        self.int_char = dict(enumerate(self.chars))
        self.char_int = {ch : i for i, ch in self.int_char.items()}

        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layer, dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)

        self.fc = nn.Linear(n_hidden, len(self.chars))

    def forward(self, x, hidden):
        r_output, hidden = self.lstm(x, hidden)

        out = self.dropout(r_output)
       
        out = out.contagious().view(-1, self.n_hidden)

        out = self.fc(out)

        return out, hidden
    
    def hidden (self, batch_size):
        
        weight = next(self.parameters()).data
        
        hidden = (weight.new(self.n_layer, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layer, batch_size, self.n_hidden).zero_())
        
        return hidden